# EDA completo — bbdd_filtrada.csv (Tienda 44, 2016-2017, base unida)

EDA sobre la base ya unida y filtrada a la tienda 44 (`train_2016_2017_union.csv` + `stores` + `transactions` + `items` + `holidays_events` + `oil`, filtrado a `store_nbr == 44`). Como es una sola tienda, el archivo es manejable y se carga completo (sin muestreo) para todos los números reportados — solo los gráficos de dispersión usan una muestra acotada, únicamente por legibilidad visual (no afecta las estadísticas).

## 0. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

BASE_DIR = "C:/Tesis"
PLOT_SAMPLE_SIZE = 100_000   # solo para gráficos de dispersión (muchos puntos = ilegible)
RANDOM_STATE = 42

# Paleta simple y consistente para todo el notebook
COLOR_PRINCIPAL = "#4C72B0"      # azul — series únicas (histogramas, líneas, barras simples)
COLOR_SECUNDARIO = "#DD8452"     # naranja — para comparaciones de 2 grupos
PALETA_CATEGORICA = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
"#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
MAPA_SECUENCIAL = "Blues"        # para el heatmap de correlaciones

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})


## 1. Carga de datos
Le especificamos los tipos de dato de antemano para que cargue más rápido y use menos memoria.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "bbdd_filtrada.csv")

tamano_mb = os.path.getsize(FILE_PATH) / (1024**2)
print(f"Tamaño en disco: {tamano_mb:.1f} MB")


In [ ]:
DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
    "cluster": "Int16",
    "class": "Int32",
    "perishable": "Int8",
    "transactions": "Int32",
}

df = pd.read_csv(FILE_PATH, dtype=DTYPES, parse_dates=["date"])
print(f"Filas: {len(df):,}")
print(f"Columnas: {df.shape[1]}")


## 2. Vista general

In [ ]:
df.info()


In [ ]:
df.head(10)


In [ ]:
print("Rango de fechas:", df["date"].min().date(), "->", df["date"].max().date())
print("Tiendas presentes:", df["store_nbr"].unique())
print("Productos distintos:", f"{df['item_nbr'].nunique():,}")
print(f"Memoria usada: {df.memory_usage(deep=True).sum() / (1024**2):.1f} MB")


## 3. Valores nulos
Es esperable ver nulos en `dcoilwtico` (el petróleo no se transa fines de semana/feriados), en las columnas de `holidays_events` (la mayoría de los días no son feriado) y en `onpromotion` (no todas las filas lo indican). Nulos en otras columnas sí ameritan revisión.

In [ ]:
nulos = df.isna().sum()
nulos_pct = (df.isna().mean() * 100).round(2)
resumen_nulos = pd.DataFrame({"nulos": nulos, "porcentaje": nulos_pct}).sort_values("nulos", ascending=False)
resumen_nulos


In [ ]:
con_nulos = resumen_nulos[resumen_nulos["nulos"] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(con_nulos.index[::-1], con_nulos["porcentaje"][::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("% de filas nulas")
ax.set_title("Porcentaje de nulos por columna")
plt.tight_layout()
plt.show()


## 4. Duplicados

In [ ]:
dup_id = df["id"].duplicated().sum()
dup_combo = df.duplicated(subset=["store_nbr", "item_nbr", "date"]).sum()

print("Filas con id duplicado:", dup_id)
print("Filas con combinación (tienda, producto, fecha) duplicada:", dup_combo)


## 5. Estadísticas descriptivas — variables numéricas

In [ ]:
df[["unit_sales", "transactions", "dcoilwtico", "class", "cluster"]].describe()


## 6. Variables categóricas
`store_type` y `cluster` deberían salir con un solo valor (es una sola tienda) — es un buen chequeo de que el filtro por tienda 44 quedó bien.

In [ ]:
print("store_type:", df["store_type"].unique())
print("cluster:", df["cluster"].unique())
print("city:", df["city"].unique())
print("state:", df["state"].unique())


In [ ]:
top_familias = df["family"].value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top_familias.index[::-1], top_familias.values[::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("Cantidad de filas")
ax.set_title("Top 10 familias de productos (por cantidad de registros)")
plt.tight_layout()
plt.show()


In [ ]:
print("onpromotion:")
print(df["onpromotion"].value_counts(dropna=False))
print()
print("perishable:")
print(df["perishable"].value_counts(dropna=False))
print()
print("holiday_type (solo días feriado, el resto es NaN):")
print(df["holiday_type"].value_counts(dropna=False))


## 7. Distribución de unit_sales (la variable a predecir)
`unit_sales` suele venir muy sesgada (muchas ventas chicas, pocas muy grandes), por eso miramos también su versión en escala logarítmica.

In [ ]:
df["unit_sales"].describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["unit_sales"], bins=100, color=COLOR_PRINCIPAL)
axes[0].set_title("unit_sales (escala original)")
axes[0].set_xlabel("unit_sales")
axes[0].set_ylabel("Frecuencia")

axes[1].hist(np.log1p(df["unit_sales"].clip(lower=0)), bins=100, color=COLOR_SECUNDARIO)
axes[1].set_title("log(1 + unit_sales), valores negativos recortados a 0")
axes[1].set_xlabel("log1p(unit_sales)")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(df["unit_sales"], vert=True)
ax.set_title("unit_sales — boxplot")
ax.set_ylabel("unit_sales")
plt.tight_layout()
plt.show()


In [ ]:
q1 = df["unit_sales"].quantile(0.25)
q3 = df["unit_sales"].quantile(0.75)
iqr = q3 - q1
limite_inf = q1 - 1.5 * iqr
limite_sup = q3 + 1.5 * iqr

outliers = df[(df["unit_sales"] < limite_inf) | (df["unit_sales"] > limite_sup)]
negativos = df[df["unit_sales"] < 0]

print(f"Rango normal (IQR): [{limite_inf:.2f}, {limite_sup:.2f}]")
print(f"Filas fuera de ese rango (outliers): {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")
print(f"Filas con unit_sales negativo (devoluciones): {len(negativos):,} ({len(negativos)/len(df)*100:.2f}%)")


## 8. Evolución temporal

In [ ]:
ventas_mensuales = df.groupby(df["date"].dt.to_period("M"))["unit_sales"].sum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ventas_mensuales.index.astype(str), ventas_mensuales.values, marker="o", color=COLOR_PRINCIPAL)
ax.set_title("Suma de unit_sales por mes — Tienda 44")
ax.set_xlabel("Mes")
ax.set_ylabel("Suma de unit_sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
transacciones_diarias = df.drop_duplicates(subset="date")[["date", "transactions"]].sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(transacciones_diarias["date"], transacciones_diarias["transactions"], color=COLOR_SECUNDARIO, linewidth=0.8)
ax.set_title("Transacciones diarias — Tienda 44")
ax.set_xlabel("Fecha")
ax.set_ylabel("Transacciones")
plt.tight_layout()
plt.show()


In [ ]:
precio_petroleo = df.drop_duplicates(subset="date")[["date", "dcoilwtico"]].sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(precio_petroleo["date"], precio_petroleo["dcoilwtico"], color="#55A868", linewidth=0.8)
ax.set_title("Precio del petróleo (dcoilwtico) — 2016-2017")
ax.set_xlabel("Fecha")
ax.set_ylabel("USD por barril")
plt.tight_layout()
plt.show()


## 9. Relación de las variables con unit_sales

### unit_sales promedio por familia de producto (top 10)

In [ ]:
top10_familias = df["family"].value_counts().head(10).index
promedio_por_familia = (
    df[df["family"].isin(top10_familias)]
    .groupby("family")["unit_sales"].mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(promedio_por_familia.index[::-1], promedio_por_familia.values[::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("unit_sales promedio")
ax.set_title("unit_sales promedio por familia (top 10 familias más frecuentes)")
plt.tight_layout()
plt.show()


### unit_sales: en promoción vs. sin promoción

In [ ]:
promedio_promocion = df.groupby(df["onpromotion"].fillna("Sin dato"))["unit_sales"].mean()
print(promedio_promocion)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(promedio_promocion.index.astype(str), promedio_promocion.values,
       color=[COLOR_PRINCIPAL, COLOR_SECUNDARIO, "#8C8C8C"][:len(promedio_promocion)])
ax.set_ylabel("unit_sales promedio")
ax.set_title("unit_sales promedio según onpromotion")
plt.tight_layout()
plt.show()


### unit_sales: días feriado vs. días normales
Ojo: esto marca como feriado cualquier día donde `holiday_type` no sea nulo, sin distinguir si el feriado era local de otra ciudad o si fue `transferred` (trasladado a otra fecha) — es una simplificación, no una regla perfecta de calendario.

In [ ]:
df["es_feriado"] = df["holiday_type"].notna()

promedio_feriado = df.groupby("es_feriado")["unit_sales"].mean()
print(promedio_feriado)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Día normal", "Feriado"], promedio_feriado.values, color=[COLOR_PRINCIPAL, COLOR_SECUNDARIO])
ax.set_ylabel("unit_sales promedio")
ax.set_title("unit_sales promedio: feriado vs. día normal")
plt.tight_layout()
plt.show()


### unit_sales vs. precio del petróleo
Con más de un millón de puntos el gráfico de dispersión queda ilegible, así que acá sí usamos una muestra aleatoria (`PLOT_SAMPLE_SIZE`) — solo para que el gráfico se vea bien, no para el cálculo de la correlación (esa sí es sobre el 100% de los datos).

In [ ]:
correlacion_oil = df[["unit_sales", "dcoilwtico"]].corr().iloc[0, 1]
print(f"Correlación unit_sales vs dcoilwtico (sobre el 100% de los datos): {correlacion_oil:.4f}")

muestra_plot = df.dropna(subset=["dcoilwtico"]).sample(min(PLOT_SAMPLE_SIZE, df["dcoilwtico"].notna().sum()), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(muestra_plot["dcoilwtico"], muestra_plot["unit_sales"], alpha=0.15, s=8, color=COLOR_PRINCIPAL)
ax.set_xlabel("Precio del petróleo (dcoilwtico)")
ax.set_ylabel("unit_sales")
ax.set_title("unit_sales vs. precio del petróleo (muestra)")
plt.tight_layout()
plt.show()


### unit_sales vs. transacciones de la tienda ese día

In [ ]:
correlacion_trans = df[["unit_sales", "transactions"]].corr().iloc[0, 1]
print(f"Correlación unit_sales vs transactions (sobre el 100% de los datos): {correlacion_trans:.4f}")

muestra_plot2 = df.sample(min(PLOT_SAMPLE_SIZE, len(df)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(muestra_plot2["transactions"], muestra_plot2["unit_sales"], alpha=0.15, s=8, color=COLOR_SECUNDARIO)
ax.set_xlabel("Transacciones de la tienda ese día")
ax.set_ylabel("unit_sales")
ax.set_title("unit_sales vs. transacciones (muestra)")
plt.tight_layout()
plt.show()


### Matriz de correlaciones (variables numéricas)

In [ ]:
numericas = df[["unit_sales", "transactions", "dcoilwtico", "perishable", "cluster", "class"]]
corr = numericas.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap=MAPA_SECUENCIAL, vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Correlación")
ax.set_title("Matriz de correlaciones")
plt.tight_layout()
plt.show()


## 10. Resumen ejecutivo
Un resumen numérico de todo lo anterior, útil para pegarlo directo en la tesis (sección de resultados del EDA).

In [ ]:
print("===== RESUMEN EDA — bbdd_filtrada (Tienda 44, 2016-2017) =====")
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
print(f"Rango de fechas: {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Productos distintos: {df['item_nbr'].nunique():,}")
print(f"IDs duplicados: {dup_id} | Combinaciones duplicadas: {dup_combo}")
print()
print(f"unit_sales -> media: {df['unit_sales'].mean():.2f} | mediana: {df['unit_sales'].median():.2f} | "
      f"std: {df['unit_sales'].std():.2f} | min: {df['unit_sales'].min():.2f} | max: {df['unit_sales'].max():.2f}")
print(f"% de filas con unit_sales negativo (devoluciones): {len(negativos)/len(df)*100:.2f}%")
print(f"% de filas outlier (regla IQR): {len(outliers)/len(df)*100:.2f}%")
print()
print(f"unit_sales promedio EN promoción: {promedio_promocion.get(True, float('nan')):.2f}")
print(f"unit_sales promedio SIN promoción: {promedio_promocion.get(False, float('nan')):.2f}")
print(f"unit_sales promedio en FERIADO: {promedio_feriado.get(True, float('nan')):.2f}")
print(f"unit_sales promedio en día normal: {promedio_feriado.get(False, float('nan')):.2f}")
print()
print(f"Correlación unit_sales - dcoilwtico: {correlacion_oil:.4f}")
print(f"Correlación unit_sales - transactions: {correlacion_trans:.4f}")
print(f"Familia con mayor unit_sales promedio (top 10 más frecuentes): {promedio_por_familia.index[0]}")


---
## Notas para la tesis
- Los valores negativos de `unit_sales` son devoluciones, no errores de datos — decide si los dejas, los excluyes, o los analizas aparte según el enfoque de tu modelo.
- El marcado de `es_feriado` es una simplificación (no distingue feriados locales de otra ciudad ni `transferred`) — defínelo con más detalle si tu análisis lo necesita.
- `store_type` y `cluster` salieron con un solo valor porque es una sola tienda — es esperable, no un error.